# Customer Segmentation — E-Commerce (Online Retail II)

**Auteur** : Nasserine Benchettara — Data Scientist | Customer Analytics  
**Dataset** : [Online Retail II — UCI Machine Learning Repository](https://archive.ics.uci.edu/ml/datasets/Online+Retail+II)  
**Periode** : 2009–2011  

---

## Contexte et objectifs

Ce notebook realise une analyse complete de segmentation client sur un dataset
e-commerce B2C d'un retailer britannique.

**3 objectifs :**

| # | Objectif | Livrable |
|---|----------|----------|
| 1 | Nettoyer et preparer les donnees transactionnelles | Dataset propre |
| 2 | Construire les features comportementales (RFM + cancel rate) | master_customer.csv |
| 3 | Segmenter les clients par comportement | Segments actionables |

---

## Structure

```
1. Setup & chargement
2. Exploration initiale (EDA)
3. Nettoyage des donnees
4. Feature engineering (RFM + cancel_rate)
5. Segmentation client
6. Synthese & recommandations
```

## 1. Setup & chargement

In [ ]:
import numpy as np
import pandas as pd
import datetime as dt
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Style global
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor'  : '#F9F9F9',
    'axes.grid'       : True,
    'grid.alpha'      : 0.35,
    'axes.spines.top' : False,
    'axes.spines.right': False,
    'font.size'       : 11,
})

COLORS = ['#4ECBA4', '#7B6FF0', '#E8A44A', '#E06060', '#5A9CF7', '#B5AA9A']

In [ ]:
# Chargement du dataset
# Source : https://archive.ics.uci.edu/ml/datasets/Online+Retail+II
df = pd.read_csv('data/online_retail_II.csv', encoding='utf-8')

# Renommage pour compatibilite
df = df.rename(columns={'Customer ID': 'CustomerID'})

print(f'Shape initial : {df.shape}')
print(f'Colonnes      : {df.columns.tolist()}')
df.head(5)

---
## 2. Exploration initiale (EDA)

Avant tout nettoyage, on comprend la structure et les signaux du dataset.
L'EDA repond a 3 questions : qualite des donnees, distribution des variables,
premiers signaux business.

### 2.1 Structure et qualite

In [ ]:
print('=== Types et valeurs manquantes ===')
print(df.dtypes)
print()
print('Valeurs manquantes :')
nulls = df.isnull().sum()
pct   = (nulls / len(df) * 100).round(1)
print(pd.DataFrame({'nulls': nulls, 'pct': pct})[nulls > 0])

print(f'\nDoublons : {df.duplicated().sum()}')

> **Constat** : 22% de valeurs manquantes sur `CustomerID` et <0.1% sur `Description`.
> Ces lignes ne peuvent pas etre attribuees a un client — on les supprimera.

### 2.2 Distribution geographique

In [ ]:
# Repartition des commandes par pays
countries = df.groupby('Country')['Invoice'].nunique().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(13, 5))
countries.head(15).plot(kind='bar', ax=ax, color=COLORS[0], alpha=0.85, edgecolor='white')
ax.set_title('Top 15 pays par nombre de commandes', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Nb commandes')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('viz_01_countries.png', dpi=150, bbox_inches='tight')
plt.show()

uk_pct = countries['United Kingdom'] / countries.sum() * 100
print(f'UK represente {uk_pct:.1f}% des commandes.')

### 2.3 Volume clients, produits, transactions

In [ ]:
summary = pd.DataFrame([{
    'Transactions'       : df['Invoice'].nunique(),
    'Clients uniques'    : df['CustomerID'].nunique(),
    'Produits uniques'   : df['StockCode'].nunique(),
    'Pays'               : df['Country'].nunique(),
    'Lignes totales'     : len(df),
}])
print(summary.T.rename(columns={0:'Valeur'}).to_string())

### 2.4 Commandes annulees

In [ ]:
# Les commandes annulees commencent par 'C'
canceled = df[df['Invoice'].astype(str).str.startswith('C')]
total    = df['Invoice'].nunique()
n_cancel = canceled['Invoice'].nunique()

print(f'Commandes totales   : {total:,}')
print(f'Commandes annulees  : {n_cancel:,} ({n_cancel/total*100:.1f}%)')

# Taux d'annulation par pays
cancel_by_country = (
    canceled.groupby('Country')['Invoice'].nunique() /
    df.groupby('Country')['Invoice'].nunique()
).dropna().sort_values(ascending=False)

print('\nTop 10 pays par taux d annulation :')
print(cancel_by_country.head(10).apply(lambda x: f'{x*100:.1f}%').to_string())

---
## 3. Nettoyage des donnees

Le nettoyage suit 4 etapes dans l'ordre logique :
1. Supprimer les lignes sans client (CustomerID manquant)
2. Supprimer les doublons
3. Traiter les StockCodes speciaux (remises, frais de port)
4. Traiter les commandes annulees (quantites negatives)

### 3.1 Valeurs manquantes et doublons

In [ ]:
df_clean = df.copy()

# Suppression des lignes sans CustomerID
# Ces lignes ne peuvent etre attribuees a aucun client
before = len(df_clean)
df_clean = df_clean.dropna(subset=['CustomerID'])
print(f'Lignes supprimees (CustomerID manquant) : {before - len(df_clean):,}')

# Suppression des doublons exacts
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f'Doublons supprimes : {before - len(df_clean):,}')

print(f'Shape apres nettoyage initial : {df_clean.shape}')

### 3.2 StockCodes speciaux

In [ ]:
# Identifier les codes non-produits (contiennent uniquement des lettres)
special_codes = df_clean[
    df_clean['StockCode'].astype(str).str.contains('^[a-zA-Z]+', regex=True)
]['StockCode'].unique()

print('Codes speciaux identifies :')
for code in special_codes:
    desc = df_clean[df_clean['StockCode'] == code]['Description'].dropna()
    if len(desc) > 0:
        print(f'  {code:<15} -> {desc.iloc[0]}')

In [ ]:
# Extraire les remises et frais de port AVANT de supprimer ces lignes
# On les stocke comme features client plutot que de perdre l'information

# Remises (StockCode == 'D')
discounts = df_clean[df_clean['StockCode'] == 'D'].groupby('Invoice')['Price'].sum()
df_clean['Discount'] = df_clean['Invoice'].map(discounts).fillna(0)

# Frais de port (StockCode == 'POST')
postage = df_clean[df_clean['StockCode'] == 'POST'].groupby('Invoice')['Price'].sum()
df_clean['Postage'] = df_clean['Invoice'].map(postage).fillna(0)

print(f'Commandes avec remise  : {(df_clean["Discount"] > 0).sum():,}')
print(f'Commandes avec port    : {(df_clean["Postage"] > 0).sum():,}')

# Supprimer les lignes non-produits
df_clean = df_clean[~df_clean['StockCode'].astype(str).str.contains('^[a-zA-Z]+', regex=True)]
print(f'Shape apres suppression codes speciaux : {df_clean.shape}')

### 3.3 Commandes annulees (quantites negatives)

**Logique de traitement** : quand une commande est annulee, une ligne
avec quantite negative apparait. On cherche la contrepartie positive
correspondante pour reconstruire l'historique propre.

Trois cas possibles :
- **Aucune contrepartie** : transaction douteuse → suppression
- **Une contrepartie** : on note la quantite annulee sur la ligne originale
- **Plusieurs contreparties** : on prend la plus recente

In [ ]:
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
df_work = df_clean.copy(deep=True)
df_work['QuantityCanceled'] = 0

entry_to_remove = []
doubtfull_entry = []

for index, col in df_clean[df_clean['Quantity'] < 0].iterrows():
    # Chercher la commande originale correspondante
    mask = (
        (df_clean['CustomerID'] == col['CustomerID']) &
        (df_clean['StockCode']  == col['StockCode'])  &
        (df_clean['InvoiceDate'] < col['InvoiceDate']) &
        (df_clean['Quantity'] > 0)
    )
    df_test = df_clean[mask].copy()

    if df_test.shape[0] == 0:
        # Aucune contrepartie — transaction douteuse
        doubtfull_entry.append(index)

    elif df_test.shape[0] == 1:
        # Une seule contrepartie
        idx = df_test.index[0]
        df_work.loc[idx, 'QuantityCanceled'] = -col['Quantity']
        entry_to_remove.append(index)

    else:
        # Plusieurs contreparties — prendre la plus recente
        df_test = df_test.sort_index(ascending=False)
        for ind, val in df_test.iterrows():
            if val['Quantity'] < -col['Quantity']:
                continue
            df_work.loc[ind, 'QuantityCanceled'] = -col['Quantity']
            entry_to_remove.append(index)
            break

print(f'Lignes a supprimer (annulations avec contrepartie) : {len(entry_to_remove):,}')
print(f'Lignes douteuses (sans contrepartie)               : {len(doubtfull_entry):,}')

In [ ]:
# Suppression des lignes identifiees
df_work.drop(entry_to_remove + doubtfull_entry, axis=0, inplace=True)

# Supprimer les quantites negatives restantes
remaining = df_work[(df_work['Quantity'] < 0)]
df_work.drop(remaining.index, axis=0, inplace=True)

print(f'Shape final apres nettoyage complet : {df_work.shape}')
print(f'Reduction : {len(df) - len(df_work):,} lignes supprimees ({(len(df) - len(df_work))/len(df)*100:.1f}%)')
df_work.head(3)

In [ ]:
# Export du dataset nettoye
df_work.to_csv('data/online_retail_cleaned.csv', index=False)
print('online_retail_cleaned.csv exporte')

---
## 4. Feature Engineering

On construit les variables comportementales client.
La nouveaute par rapport a un RFM classique : on enrichit avec le **cancel_rate**
(taux d'annulation par client), un signal de satisfaction / confiance rarement utilise.

### 4.1 Pourquoi enrichir le RFM avec le cancel_rate ?

Le RFM classique mesure la valeur client (qui achete, combien, quand).
Mais il ne capte pas les signaux negatifs : un client qui annule frequemment
a un profil de risque tres different d'un client avec le meme RFM mais zero annulation.

Le `cancel_rate` = nb_commandes_annulees / nb_commandes_totales permet de
distinguer ces deux profils et d'eviter de cibler des clients problematiques
avec des offres premium.

### 4.2 Variables RFM

In [ ]:
TODAY = df_work['InvoiceDate'].max() + pd.Timedelta(days=1)

# Revenue par transaction
df_work['Revenue'] = df_work['Quantity'] * df_work['Price']

rfm = df_work.groupby('CustomerID').agg(
    Recency   = ('InvoiceDate',    lambda x: (TODAY - x.max()).days),
    Frequency = ('Invoice',        'nunique'),
    Monetary  = ('Revenue',        'sum'),
    AvgBasket = ('Revenue',        'mean'),
    FirstPurchase = ('InvoiceDate', 'min'),
).reset_index()

rfm['CustomerAgeDays'] = (TODAY - rfm['FirstPurchase']).dt.days
rfm = rfm.drop(columns=['FirstPurchase'])

print(f'Table RFM : {rfm.shape}')
print(rfm.describe().round(2))

### 4.3 Cancel rate — enrichissement cle

In [ ]:
# Identifier les commandes annulees
df_work['is_canceled'] = df_work['Invoice'].astype(str).str.startswith('C').astype(int)

cancel_features = df_work.groupby('CustomerID').agg(
    nb_orders          = ('Invoice', 'nunique'),
    nb_orders_canceled = ('is_canceled', 'sum'),
).reset_index()

# Taux d'annulation : proportion de commandes annulees
cancel_features['cancel_rate'] = (
    cancel_features['nb_orders_canceled'] / cancel_features['nb_orders']
).fillna(0)

print('Distribution du cancel_rate :')
print(cancel_features['cancel_rate'].describe().round(3))
print(f'\nClients avec 0% d annulation : {(cancel_features["cancel_rate"]==0).sum():,}')
print(f'Clients avec >50% d annulation: {(cancel_features["cancel_rate"]>0.5).sum():,}')

### 4.4 Variables de comportement geographique et produit

In [ ]:
# Pays principal du client
country_features = df_work.groupby('CustomerID').agg(
    country = ('Country', lambda x: x.value_counts().idxmax()),
).reset_index()

# Diversite produit : nb de categories de produits distinctes
product_features = df_work.groupby('CustomerID').agg(
    nb_products_distinct = ('StockCode', 'nunique'),
).reset_index()

print('Features construites :')
print(f'  Pays principal   : {country_features.shape}')
print(f'  Diversite produit: {product_features.shape}')

### 4.5 Fusion — table master_customer

In [ ]:
master = (
    rfm
    .merge(cancel_features[['CustomerID','cancel_rate']], on='CustomerID')
    .merge(country_features, on='CustomerID')
    .merge(product_features, on='CustomerID')
)

print(f'Table master_customer : {master.shape[0]:,} clients x {master.shape[1]} variables')
print(f'Valeurs manquantes    : {master.isnull().sum().sum()}')
print(f'\nColonnes : {master.columns.tolist()}')
master.head(3)

### 4.6 Visualisation des features construites

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

# 1 - Recency
axes[0].hist(master['Recency'], bins=40, color=COLORS[0], alpha=0.85, edgecolor='white')
axes[0].axvline(master['Recency'].median(), color='#E06060', linewidth=2, linestyle='--',
                label=f'Mediane : {master["Recency"].median():.0f}j')
axes[0].set_title('Recency (jours depuis dernier achat)', fontweight='bold')
axes[0].legend()

# 2 - Frequency
axes[1].hist(master['Frequency'], bins=40, color=COLORS[1], alpha=0.85, edgecolor='white')
axes[1].set_title('Frequency (nb transactions)', fontweight='bold')
axes[1].set_xlim(0, master['Frequency'].quantile(0.99))

# 3 - Monetary
axes[2].hist(master['Monetary'].clip(upper=master['Monetary'].quantile(0.99)),
             bins=40, color=COLORS[2], alpha=0.85, edgecolor='white')
axes[2].set_title('Monetary — Revenue total client', fontweight='bold')

# 4 - Cancel rate
axes[3].hist(master['cancel_rate'], bins=30, color=COLORS[3], alpha=0.85, edgecolor='white')
axes[3].set_title('Cancel Rate (taux d annulation)', fontweight='bold')
axes[3].set_xlabel('Proportion de commandes annulees')

# 5 - Nb produits distincts
axes[4].hist(master['nb_products_distinct'].clip(upper=master['nb_products_distinct'].quantile(0.99)),
             bins=40, color=COLORS[4], alpha=0.85, edgecolor='white')
axes[4].set_title('Diversite produit (nb SKUs distincts)', fontweight='bold')

# 6 - Top 10 pays
top_countries = master['country'].value_counts().head(10)
axes[5].barh(top_countries.index[::-1], top_countries.values[::-1],
             color=COLORS[0], alpha=0.85, edgecolor='white')
axes[5].set_title('Top 10 pays des clients', fontweight='bold')

plt.suptitle('Feature Engineering — Vue d ensemble des variables client',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('viz_02_features.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Segmentation Client

### Choix de l'approche

On utilise **K-Means** sur les variables RFM + cancel_rate.
On valide le choix de k avec deux metriques complementaires :
- **Elbow method** : inertie (somme des distances au centroide)
- **Silhouette score** : qualite de la separation entre groupes (entre -1 et 1)

On choisit k sur les deux signaux combine, pas uniquement le meilleur silhouette
s'il ne produit pas de groupes interpretables business.

### 5.1 Preparation des features pour K-Means

In [ ]:
# Variables retenues pour le clustering
# On exclut les variables categorielles (country)
# On exclut CustomerID (identifiant, pas une feature)
FEATURES = ['Recency', 'Frequency', 'Monetary', 'AvgBasket',
            'cancel_rate', 'nb_products_distinct', 'CustomerAgeDays']

X = master[FEATURES].copy()

# Capper les valeurs extremes avant normalisation
# Les outliers (gros comptes B2B) perturbent K-Means
for col in ['Monetary', 'Frequency', 'nb_products_distinct']:
    cap = X[col].quantile(0.99)
    X[col] = X[col].clip(upper=cap)

# Normalisation : K-Means calcule des distances euclidiennes
# Sans normalisation, Monetary (0-50000) ecrase Recency (0-700)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'Features pour K-Means : {FEATURES}')
print(f'Shape normalise : {X_scaled.shape}')

### 5.2 Choix du nombre de clusters

In [ ]:
inertias    = []
silhouettes = []
K_range     = range(2, 9)

for k in K_range:
    km     = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))
    print(f'k={k} | inertie={km.inertia_:,.0f} | silhouette={silhouette_score(X_scaled, labels):.4f}')

best_k_sil = list(K_range)[int(np.argmax(silhouettes))]
print(f'\nMeilleur k selon silhouette : k={best_k_sil}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Elbow Method
axes[0].plot(list(K_range), inertias, marker='o', linewidth=2,
             color=COLORS[1], markersize=7)
axes[0].set_title('Elbow Method\nChercher le coude (inflexion)', fontweight='bold')
axes[0].set_xlabel('Nombre de clusters (k)')
axes[0].set_ylabel('Inertie')

# Silhouette Score
axes[1].plot(list(K_range), silhouettes, marker='o', linewidth=2,
             color=COLORS[0], markersize=7)
axes[1].set_title('Silhouette Score par k\nPlus haut = meilleure separation', fontweight='bold')
axes[1].set_xlabel('Nombre de clusters (k)')
axes[1].set_ylabel('Score (entre -1 et 1)')
axes[1].axvline(best_k_sil, color=COLORS[2], linestyle='--', linewidth=1.5,
                label=f'k={best_k_sil} meilleur score')
axes[1].legend()

plt.tight_layout()
plt.savefig('viz_03_elbow.png', dpi=150, bbox_inches='tight')
plt.show()

print('Note : on regarde les deux graphiques ensemble.')
print('Si le k optimal silhouette ne produit pas de segments interpretables,')
print('on peut choisir un k adjacent plus riche business.')

### 5.3 Fit final et profiling

In [ ]:
# On choisit k=4 : bon compromis silhouette + lisibilite business
# A ajuster selon vos resultats
K_FINAL = 4

km_final = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10)
master['cluster'] = km_final.fit_predict(X_scaled)

print(f'Silhouette score final (k={K_FINAL}) : {silhouette_score(X_scaled, master["cluster"]):.4f}')
print()
print('Repartition des clusters :')
print(master['cluster'].value_counts().sort_index())

### 5.4 Profil des clusters

In [ ]:
profil = master.groupby('cluster').agg(
    nb_clients          = ('CustomerID',         'count'),
    recency_moy         = ('Recency',             'mean'),
    frequency_moy       = ('Frequency',           'mean'),
    monetary_moy        = ('Monetary',             'mean'),
    avg_basket_moy      = ('AvgBasket',            'mean'),
    cancel_rate_moy     = ('cancel_rate',          'mean'),
    nb_produits_moy     = ('nb_products_distinct', 'mean'),
    ca_total            = ('Monetary',             'sum'),
).round(2)

profil['pct_base'] = (profil['nb_clients'] / len(master) * 100).round(1)
profil['pct_ca']   = (profil['ca_total'] / master['Monetary'].sum() * 100).round(1)

print('=' * 85)
print('PROFIL DES CLUSTERS')
print('=' * 85)
print(profil[['nb_clients','pct_base','recency_moy','frequency_moy',
              'monetary_moy','cancel_rate_moy','ca_total','pct_ca']].to_string())
print('=' * 85)

### 5.5 Nommage des segments

In [ ]:
# Nommer les segments selon leur profil
# A adapter selon vos resultats reels

# Logique de nommage basee sur les caracteristiques dominantes :
# - Recency faible + Frequency haute + Monetary haute + cancel_rate faible -> Champions
# - Recency moyenne + Frequency moyenne -> Fideles
# - Recency haute (inactif) -> Dormants / A risque
# - Cancel_rate eleve -> Insatisfaits

def nommer_segment(row):
    if row['cancel_rate_moy'] > 0.3:
        return 'Insatisfait'
    elif row['recency_moy'] < 50 and row['frequency_moy'] > 5:
        return 'Champion'
    elif row['recency_moy'] < 150 and row['frequency_moy'] > 2:
        return 'Fidele'
    else:
        return 'Dormant'

profil['nom_segment'] = profil.apply(nommer_segment, axis=1)
mapping = profil['nom_segment'].to_dict()
master['segment'] = master['cluster'].map(mapping)

print('Segments identifies :')
for c, nom in mapping.items():
    n = (master['cluster']==c).sum()
    print(f'  Cluster {c} -> {nom:<15} ({n:,} clients)')

### 5.6 Visualisation des segments

In [ ]:
seg_names  = list(master['segment'].unique())
seg_colors = dict(zip(seg_names, COLORS[:len(seg_names)]))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

# 1 - Nombre de clients
nb = master.groupby('segment')['CustomerID'].count().sort_values(ascending=False)
axes[0].bar(nb.index, nb.values,
            color=[seg_colors.get(s, '#ccc') for s in nb.index],
            alpha=0.85, edgecolor='white')
axes[0].set_title('Nombre de clients par segment', fontweight='bold')
for i, v in enumerate(nb.values):
    axes[0].text(i, v+10, f'{v:,}\n({v/len(master)*100:.1f}%)',
                 ha='center', fontsize=10, fontweight='bold')

# 2 - Revenue moyen
rev = master.groupby('segment')['Monetary'].mean().sort_values(ascending=False)
axes[1].bar(rev.index, rev.values,
            color=[seg_colors.get(s, '#ccc') for s in rev.index],
            alpha=0.85, edgecolor='white')
axes[1].set_title('Revenue moyen par client (euros)', fontweight='bold')
for i, v in enumerate(rev.values):
    axes[1].text(i, v+10, f'{v:.0f}e', ha='center', fontsize=11, fontweight='bold')

# 3 - Cancel rate moyen
cr = master.groupby('segment')['cancel_rate'].mean().sort_values(ascending=False)
axes[2].bar(cr.index, cr.values * 100,
            color=[seg_colors.get(s, '#ccc') for s in cr.index],
            alpha=0.85, edgecolor='white')
axes[2].set_title('Taux d annulation moyen (%)', fontweight='bold')
axes[2].set_ylabel('%')
for i, v in enumerate(cr.values):
    axes[2].text(i, v*100+0.3, f'{v*100:.1f}%', ha='center', fontsize=11, fontweight='bold')

# 4 - Scatter Recency vs Monetary
for seg in seg_names:
    sub = master[master['segment'] == seg]
    axes[3].scatter(sub['Recency'], sub['Monetary'].clip(upper=sub['Monetary'].quantile(0.95)),
                    c=seg_colors.get(seg, '#ccc'), label=seg,
                    alpha=0.4, s=10, edgecolors='none')
axes[3].set_xlabel('Recency (jours)')
axes[3].set_ylabel('Revenue total (euros)')
axes[3].set_title('Recency vs Monetary par segment', fontweight='bold')
axes[3].legend(markerscale=3, fontsize=9)

plt.suptitle('Segmentation Client — Profil des segments',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('viz_04_segments.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.7 Apport du cancel_rate dans la segmentation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Cancel rate par segment
cr_seg = master.groupby('segment')['cancel_rate'].mean().sort_values(ascending=False)
axes[0].barh(cr_seg.index[::-1], cr_seg.values[::-1] * 100,
             color=[seg_colors.get(s,'#ccc') for s in cr_seg.index[::-1]],
             alpha=0.85, edgecolor='white')
axes[0].set_title('Taux d annulation moyen par segment\n'
                  '(apport du cancel_rate vs RFM classique)', fontweight='bold')
axes[0].set_xlabel('%')

# Revenue vs cancel rate (scatter)
for seg in seg_names:
    sub = master[master['segment'] == seg]
    axes[1].scatter(sub['cancel_rate'] * 100, sub['Monetary'].clip(upper=sub['Monetary'].quantile(0.95)),
                    c=seg_colors.get(seg,'#ccc'), label=seg,
                    alpha=0.4, s=15, edgecolors='none')
axes[1].set_xlabel('Cancel rate (%)')
axes[1].set_ylabel('Revenue total (euros)')
axes[1].set_title('Revenue vs Cancel Rate\n'
                  'Signal complementaire au RFM', fontweight='bold')
axes[1].legend(markerscale=2, fontsize=9)

plt.tight_layout()
plt.savefig('viz_05_cancel_rate.png', dpi=150, bbox_inches='tight')
plt.show()

print('Insight : le cancel_rate permet de distinguer les clients insatisfaits')
print('des clients simplement dormants — deux profils avec des actions differentes.')

---
## 6. Synthese & Recommandations

In [ ]:
print('=' * 70)
print('SYNTHESE DE LA SEGMENTATION CLIENT')
print('=' * 70)

print()
print('DATASET')
print(f'  Transactions initiales  : {len(df):,}')
print(f'  Apres nettoyage         : {len(df_work):,}')
print(f'  Clients segmentes       : {len(master):,}')

print()
print('SEGMENTS')
for seg in master['segment'].value_counts().index:
    sub    = master[master['segment'] == seg]
    pct_ca = sub['Monetary'].sum() / master['Monetary'].sum() * 100
    print(f'  {seg:<15} : {len(sub):>5} clients | '
          f'rev moy {sub["Monetary"].mean():>7.0f}e | '
          f'cancel {sub["cancel_rate"].mean()*100:.1f}% | '
          f'{pct_ca:.1f}% du CA')

print()
print('RECOMMANDATIONS')
print()
print('  Champions      : programme VIP, acces exclusif, offres avant-premiere')
print('  Fideles        : maintenir l engagement, cross-sell produits complementaires')
print('  Dormants       : campagne de reactivation J+90, offre incitative')
print('  Insatisfaits   : investigation SAV, pas de sollicitation marketing avant resolution')
print()
print('APPORT DU CANCEL_RATE')
print('  Sans cancel_rate : Insatisfaits et Dormants seraient regroupes')
print('  Avec cancel_rate : deux actions differentes — reactivation vs resolution SAV')
print('=' * 70)

In [ ]:
# Export final
master.to_csv('data/master_customer_segmented.csv', index=False)
print('master_customer_segmented.csv exporte')
print(f'{len(master):,} clients x {master.shape[1]} colonnes')

---

## Fichiers produits

| Fichier | Description |
|---------|-------------|
| `data/online_retail_cleaned.csv` | Dataset nettoye |
| `data/master_customer_segmented.csv` | Table client enrichie + segment |
| `viz_01_countries.png` | Distribution geographique |
| `viz_02_features.png` | Vue d ensemble des features |
| `viz_03_elbow.png` | Elbow + silhouette |
| `viz_04_segments.png` | Profil des 4 segments |
| `viz_05_cancel_rate.png` | Apport du cancel_rate |

---

## Stack technique

```
Python 3.10+
pandas, numpy — manipulation des donnees
matplotlib, seaborn, plotly — visualisations
scikit-learn — StandardScaler, KMeans, silhouette_score
```

---

*Nasserine Benchettara — Data Scientist | Customer Analytics*